# Timme et al. (2016) on CRCNS hc-8: complexity of the culture recordings

Paper: Timme et al. (2016), Front. Physiol. 7:425, doi:10.3389/fphys.2016.00425; claims file
`claims/crcns-hc-8/10.3389-fphys.2016.00425.yaml` (version of record, PMC5037237, with the
supplement). Dataset: CRCNS hc-8, doi:10.6080/K0PC308P, `hc-8/live/Data.zip` (MD5
`bd4e2a8685df3e55e1a3c04fbe09d9ce` in the data set's `checksums.md5`), version `unversioned`: CRCNS
data sets carry no versions and this is the only published copy (see `culture_criticality.ipynb`).

Complexity (Methods 2.9, `complexity.m` of the NCC toolbox): on the avalanche bins of the rebinned
first 59 min, with the sub-sampling correction (the integration curve of data whose neurons are
each permuted in time is subtracted, and the curve is cut at the first peak of I(k)/(k-1)). It uses
100 random subsets per subset size, as Methods 2.9 give, and runs on 40 of the 435 recordings drawn
with a fixed seed, to fit the time limit of a notebook. A first version used 10 subsets on 180
recordings; the verifier found the cut unstable at 10 subsets, and the spread of the estimate over
repeated draws is shown below at 10 and 100 subsets. Two variants are shown: the corrected curve without the cut, and
no correction. The randomized data are the same as in `culture_criticality.ipynb` (same
generators).

| claim | here |
|---|---|
| fig10a-complexity-randomization | tested |
| figs8-complexity-susceptibility-correlation | tested |
| figs9-complexity-fit-range-correlation | tested |
| fig12a-swap-preserves-complexity | tested |
| fig12b-shuffle-lowers-complexity | tested |

In [ ]:
import os
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(v, "1")
import time, resource
import multiprocessing as mp
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import hc8

SEED = 20161          # the same seed as culture_criticality.ipynb, so the randomized data are the same
N_SUBSETS = 100       # Methods 2.9: "the lesser of either all possible permutations of neurons or 100"
SUBSET = int(os.environ.get("HC8_SUBSET", "0"))   # only while developing
names = hc8.recording_names()
# 40 of the 435 recordings, drawn with a fixed seed: with 100 subsets per size one recording (8
# conditions) took 50 s of wall time with 4 processes in development, so all 435 would take about
# 6 h, over the 60 min limit of a notebook
N_RECORDINGS = 40
idx = sorted(np.random.default_rng([SEED, 0]).choice(len(names), size=N_RECORDINGS, replace=False).tolist())
if SUBSET:
    idx = idx[:SUBSET]
t0 = time.time()
with mp.get_context("spawn").Pool(4) as pool:
    recs = list(pool.imap(hc8.complexity_record, [(i, names[i], SEED, N_SUBSETS) for i in idx], chunksize=1))
cx = pd.DataFrame(recs)
print(f"{len(cx)} recordings in {(time.time() - t0) / 60:.1f} min; peak memory of this process "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")
KINDS = ["real"] + hc8.RANDOMIZATIONS

In [ ]:
print("recordings:", len(cx), "| cultures:", cx.culture.nunique(), "| neurons analysed:", int(cx.n_neurons.sum()))
print("avalanche bins per recording (real): %d to %d" % (cx.nbins_real.min(), cx.nbins_real.max()))
for k in KINDS:
    c = cx[f"C_{k}"]
    print(f"{k:10s} complexity (bits/neuron) median {c.median():.5f}, range {c.min():.5f} to {c.max():.5f}; "
          f"subset sizes kept by the cut: median {cx[f'kkept_{k}'].median():.0f} of {cx[f'N_{k}'].median():.0f}; "
          f"NaN {int(c.isna().sum())}")
print("neurons left without a spike by shuffling (dropped):", int((cx.N_shuffle < cx.n_neurons).sum()), "recordings")
incl_a = (cx.n_size_cut_values >= 1) & (cx.n_dur_cut_values >= 1) & (cx.n_profiles >= 2)
print("recordings kept under exclusion reading (a):", int(incl_a.sum()), "of", len(cx))

## Spread of the estimate over repeated draws

The verifier of the first version asked how much the toolbox's complexity (correction and cut)
varies with the random draws (the subsets and the time permutation of the correction). The real
data of 10 of the recordings above are run three times with different generators, at 10 and at 100
subsets per size. No claim is decided here.

In [ ]:
t1 = time.time()
with mp.get_context("spawn").Pool(4) as pool:
    parts = pool.map(hc8.complexity_seeds, [(i, names[i], SEED, (10, N_SUBSETS), 3) for i in idx[:10]], chunksize=1)
sd = pd.DataFrame([r for part in parts for r in part])
print(f"{sd.name.nunique()} recordings x 3 draws x 2 subset numbers in {(time.time() - t1) / 60:.1f} min")
for ns, g in sd.groupby("n_subsets"):
    per = g.groupby("name").agg(C_min=("C", "min"), C_max=("C", "max"), C_mean=("C", "mean"),
                                k_min=("k_kept", "min"), k_max=("k_kept", "max"), N=("N", "first"),
                                Cuncut_min=("Cuncut", "min"), Cuncut_max=("Cuncut", "max"))
    per["C_cv"] = g.groupby("name").C.std() / per.C_mean
    print(f"\n{ns} subsets per size, per recording (complexity in bits/neuron):")
    print(per.to_string(float_format=lambda v: f"{v:.5f}"))
    print(f"median over recordings of the coefficient of variation of the cut complexity: {per.C_cv.median():.2f}; "
          f"of max/min: {(per.C_max / per.C_min).median():.2f}")
    print(f"subset sizes kept by the cut, all runs: median {g.k_kept.median():.0f}, range {g.k_kept.min()} to {g.k_kept.max()} "
          f"(of N {g.N.min()} to {g.N.max()}); kept/N median {(g.k_kept / g.N).median():.2f}")


## fig10a-complexity-randomization

> Culture complexity decreases under randomization, most under the strongest forms such as Poisson
> randomization, with the exception of spike swapping.

Rank sum test of the real data against each randomization across recordings, false discovery rate
over the seven comparisons of the panel, Benjamini-Hochberg (BH) or Benjamini-Yekutieli (BY)
(claims file, common detail 3); the family of all three panels of Figure 10 is a variant.

In [ ]:
def panel(prefix, rows=incl_a, kinds=hc8.RANDOMIZATIONS, real_col=None):
    out = []
    x = cx[real_col or f"{prefix}_real"][rows].dropna()
    for k in kinds:
        y = cx[f"{prefix}_{k}"][rows].dropna()
        p = stats.mannwhitneyu(x, y, alternative="two-sided").pvalue
        out.append(dict(randomization=k, real_median=x.median(), rand_median=y.median(),
                        change=y.median() - x.median(), rel_change=(y.median() - x.median()) / x.median(), p=p))
    t = pd.DataFrame(out).set_index("randomization")
    t["p_bh"] = hc8.fdr(t.p.values, "bh"); t["p_by"] = hc8.fdr(t.p.values, "by")
    return t

fmt = lambda v: f"{v:.3g}"
NOT_SWAP = ["jitter1", "jitter10", "jitter100", "wrap", "poisson", "shuffle"]
for prefix, lab in (("C", "complexity.m (correction and cut)"), ("Cuncut", "variant: correction, no cut"), ("Craw", "variant: no correction")):
    t = panel(prefix)
    print(lab); print(t.to_string(float_format=fmt))
    sub = t.loc[NOT_SWAP]
    dec = sub.change.sort_values()
    for c in ("p_bh", "p_by"):
        ok = list(sub.index[(sub.change < 0) & (sub[c] < 0.05)])
        print(f"  {c}: lower with corrected p < 0.05 for {len(ok)} of 6 non-swap randomizations: {ok}")
    print("  two largest median decreases:", list(dec.index[:2]), "| Poisson among them:", "poisson" in list(dec.index[:2]), "\n")
    if prefix == "C":
        t10a = t
fam = pd.concat([t10a, panel("chi_a", real_col="chi_a"), panel("abs_curv", real_col="abs_curv")], keys=["C", "chi_a", "abs_curv"])
fam["p_bh_fig"] = hc8.fdr(fam.p.values, "bh"); fam["p_by_fig"] = hc8.fdr(fam.p.values, "by")
print("whole-figure family (Figure 10 A to C), complexity rows:")
print(fam.loc["C", ["change", "p", "p_bh_fig", "p_by_fig"]].to_string(float_format=fmt))
print("\nall recordings (exclusion reading b):")
print(panel("C", rows=slice(None))[["change", "p_bh", "p_by"]].to_string(float_format=fmt))
print("\nper culture, fraction of recordings with lower complexity under Poisson:",
      (cx.C_poisson < cx.C_real)[incl_a].groupby(cx.culture[incl_a]).mean().round(2).to_dict())
fig, ax = plt.subplots(figsize=(7, 3))
ax.boxplot([cx[f"C_{k}"][incl_a].dropna() for k in KINDS], whis=(0, 100)); ax.set_xticks(range(1, len(KINDS) + 1), KINDS); ax.set_yscale("symlog", linthresh=1e-4)
ax.set_ylabel("complexity (bits/neuron)"); plt.xticks(rotation=30); plt.tight_layout(); plt.show()
print("plotted range: %.5f to %.5f" % (min(cx[f'C_{k}'].min() for k in KINDS), max(cx[f'C_{k}'].max() for k in KINDS)))

**Result: not reproduced.** Over 40 recordings with 100 subsets per size, the toolbox's complexity (correction and cut) has median 0.00223 bits/neuron in the real data. 10-bin and 100-bin jitter, wrap and Poisson lower it with Benjamini-Hochberg corrected p from 0.0372 to 1.16e-06, and Poisson and 100-bin jitter give the two largest decreases. 1-bin jitter lowers the median to 0.00164 without significance (corrected p 0.836), and shuffling raises it to 0.176 (corrected p 2.85e-13), against the claim; under Benjamini-Yekutieli three of the six are significant. The cut keeps a median of 10 of 90 subset sizes for the real data and 74 for shuffled data. Without the cut shuffling lowers complexity too, but 1-bin and 10-bin jitter are not significant (corrected p 0.43 and 0.303); without the correction shuffling raises it.

## figs8-complexity-susceptibility-correlation

> Across culture recordings, complexity and susceptibility are positively correlated (correlation 0.23).

In [ ]:
r_ = cx[incl_a]
for col, lab in (("chi_a", "susceptibility reading (a) var(A/N)"), ("chi_b", "reading (b) N var(A/N)")):
    for cc, clab in (("C_real", "complexity.m"), ("Cuncut_real", "variant no cut"), ("Craw_real", "variant no correction")):
        ok = r_[[cc, col]].dropna()
        pr = stats.pearsonr(ok[cc], ok[col]); sr = stats.spearmanr(ok[cc], ok[col])
        print(f"{lab}, {clab}: Pearson r = {pr.statistic:.3f} (p = {pr.pvalue:.2g}), Spearman {sr.statistic:.3f}, n = {len(ok)}")
print("expected 0.23 +/- 0.023")

**Result: not reproduced.** Over 40 recordings, the Pearson correlation of the toolbox's complexity with the susceptibility is 0.902 with Eq. 14 and 0.905 with N var(A/N) (Spearman 0.724 and 0.745): positive, as stated, but far outside 0.23 +/- 0.023. Without the cut it is -0.605 and -0.592, and without the correction 0.267 and 0.446.

## figs9-complexity-fit-range-correlation

> Across culture recordings, complexity is positively correlated with the size fit range (0.296)
> and the duration fit range (0.197).

In [ ]:
for col, exp in (("s_range", 0.296), ("d_range", 0.197)):
    for cc, clab in (("C_real", "complexity.m"), ("Cuncut_real", "variant no cut"), ("Craw_real", "variant no correction")):
        ok = r_[[cc, col]].dropna()
        pr = stats.pearsonr(ok[cc], ok[col]); sr = stats.spearmanr(ok[cc], ok[col])
        print(f"{col}, {clab}: Pearson r = {pr.statistic:.3f} (p = {pr.pvalue:.2g}), Spearman {sr.statistic:.3f}, n = {len(ok)}; expected {exp} +/- {exp / 10:.4f}")

**Result: not reproduced.** Over 40 recordings, the Pearson correlations of the toolbox's complexity with the size and duration fit ranges are 0.060 (p 0.72) and -0.107 (p 0.51), against 0.296 and 0.197. Without the cut they are 0.052 and 0.276, and without the correction 0.369 and 0.038; no variant matches both.

## fig12a-swap-preserves-complexity

> Culture complexity does not change substantially under spike swapping, and real and swapped
> complexity are strongly correlated with most recordings near the equality line.

In [ ]:
for prefix, lab in (("C", "complexity.m"), ("Cuncut", "variant no cut"), ("Craw", "variant no correction")):
    t = panel(prefix)
    a, b = r_[f"{prefix}_real"], r_[f"{prefix}_swap"]
    ok = a.notna() & b.notna()
    pr = stats.pearsonr(a[ok], b[ok]); sr = stats.spearmanr(a[ok], b[ok]); w = stats.wilcoxon(a[ok], b[ok])
    rel = ((b - a) / a)[ok]
    print(f"{lab}: rank sum p = {t.p['swap']:.3g}, BH {t.p_bh['swap']:.3g}, BY {t.p_by['swap']:.3g}; median change {t.change['swap']:.3g} "
          f"(relative {t.rel_change['swap']:.3f}); median per-recording relative change {rel.median():.3f}; "
          f"Pearson r {pr.statistic:.3f}, Spearman {sr.statistic:.3f}; signed-rank p {w.pvalue:.3g}; n {int(ok.sum())}")
fig, ax = plt.subplots(1, 2, figsize=(8, 3.5))
for a_, k in zip(ax, ("swap", "shuffle")):
    a_.scatter(r_.C_real, r_[f"C_{k}"], s=4); m = max(r_.C_real.max(), r_[f"C_{k}"].max())
    a_.plot([0, m], [0, m], "k--", lw=.8); a_.set_xlabel("real"); a_.set_ylabel(k)
plt.tight_layout(); plt.show()

**Result: reproduced.** By the paper's rank sum test, spike swapping does not change the toolbox's complexity significantly (median 0.00223 against 0.00265, corrected p 0.836), and real and swapped complexity correlate with Pearson r 0.775, above the 0.7 reading of strong (Spearman 0.533, below it). The median per-recording relative change is -0.089. Both variants agree: without the cut p 0.927 and r 0.989, without the correction p 0.958 and r 0.998. The test uses 40 recordings, and the cut complexity of one recording has a median coefficient of variation of 0.78 over repeated draws at 100 subsets (spread section above), so r passes the 0.7 reading by 0.075.

## fig12b-shuffle-lowers-complexity

> Culture complexity decreases somewhat under neuron shuffling; real and shuffled complexity are
> strongly correlated, but real complexity tends to be greater.

Shuffling as Methods 2.5, which is what the toolbox's `swap2` does, so the variant the claims file
names is the same procedure.

In [ ]:
for prefix, lab in (("C", "complexity.m"), ("Cuncut", "variant no cut"), ("Craw", "variant no correction")):
    t = panel(prefix)
    a, b = r_[f"{prefix}_real"], r_[f"{prefix}_shuffle"]
    ok = a.notna() & b.notna()
    pr = stats.pearsonr(a[ok], b[ok]); sr = stats.spearmanr(a[ok], b[ok])
    print(f"{lab}: real > shuffled in {float((a[ok] > b[ok]).mean()):.3f} of {int(ok.sum())} recordings; rank sum BH {t.p_bh['shuffle']:.3g}, "
          f"BY {t.p_by['shuffle']:.3g}; median change {t.change['shuffle']:.3g}; Pearson r {pr.statistic:.3f}, Spearman {sr.statistic:.3f}")
print("per culture, fraction with real > shuffled (complexity.m):",
      (r_.C_real > r_.C_shuffle).groupby(r_.culture).mean().round(2).to_dict())

**Result: not reproduced.** With the toolbox's complexity, real complexity exceeds shuffled complexity in none of the 40 recordings (median change +0.174, corrected p 2.85e-13) and Pearson r is 0.431. Without the cut the claim holds: real exceeds shuffled in all 40 recordings, corrected p 2.97e-07, r 0.902. Without the correction real is never greater.

## Summary

| claim | outcome |
|---|---|
| fig10a-complexity-randomization | not reproduced |
| figs8-complexity-susceptibility-correlation | not reproduced |
| figs9-complexity-fit-range-correlation | not reproduced |
| fig12a-swap-preserves-complexity | reproduced |
| fig12b-shuffle-lowers-complexity | not reproduced |

Differences from the paper's method: 40 of the 435 recordings, drawn with a fixed seed. The 100
subsets per size of Methods 2.9 took 34.2 min for 40 recordings with 4 processes, so all 435 would
exceed the 60 min limit of a notebook. The first version used 10 subsets on 180 recordings.

With 100 subsets the toolbox's cut at the first peak of I(k)/(k-1) still falls early: it keeps a
median of 10 of 90 subset sizes in the real data and 74 in shuffled data. Over three draws on 10
recordings the cut complexity has a median coefficient of variation of 0.78 at 100 subsets and 0.86
at 10, and the cut keeps a median of 8 and 7 sizes. Shuffling then raises complexity, against
Figures 10A and 12B, and without the cut it lowers it in every recording. Spike swapping leaves
complexity unchanged under every variant, as the paper says.

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.